In [1]:
#use "../lib/nb.ml";;
run_notebook "Types.ipynb"

- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()
- : unit = ()
val lib_dir : string = "../lib"
val loaded_libs : (string, unit) Hashtbl.t = <abstr>
val load_lib : string -> unit = <fun>


- : unit = ()


# Converting an <span style="font-variant:small-caps;">Nfa</span> into a <span style="font-variant:small-caps;">Dfa</span>

In this notebook we show how an <span style="font-variant:small-caps;">Nfa</span>
$$ F = \langle Q, \Sigma, \delta, q_0, A \rangle $$
can be transformed into a <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ such that both finite state machines accept the
same language, that is we have
$$ L(F) = L\bigl(\texttt{det}(F)\bigr). $$
The idea behind this transformation is that the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ has to 
compute the set of all states that the <span style="font-variant:small-caps;">Nfa</span> $F$ could be in. 
Hence the states of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ are 
**sets** of states of the <span style="font-variant:small-caps;">Nfa</span> $F$.  A set of these states contains all those states that the <span style="font-variant:small-caps;">Nfa</span> 
$F$ could have reached.  Furthermore, a set $M$ of states of the <span style="font-variant:small-caps;">Nfa</span> $F$ is an accepting state of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ if the set $M$ contains an accepting state of the <span style="font-variant:small-caps;">Nfa</span> $F$.

## Declaring the Necessary Types

The types used in this notebook are defined in the notebook `Types.ipynb`, since they are shared with other
notebooks.  The directive `#show_type` displays their definitions.  Since OCaml does not allow Greek letters in
identifiers, the transition function $\delta$ is called `delta` and the alphabet $\Sigma$ is called `sigma`.

In Python, `State` is declared as a *type variable* via `TypeVar('State')`.  In OCaml, the types of finite state
machines have a *type parameter* `'q`, which is the type of the states.  For example, `string nfa` is the type of
an <span style="font-variant:small-caps;">Nfa</span> whose states are strings.  Type variables like `'q` need not be declared.

The transitions of an <span style="font-variant:small-caps;">Nfa</span> are labelled with a character or with
$\varepsilon$.  Python uses the string `'𝜀'` for $\varepsilon$, while OCaml uses the type `label`.

In [2]:
#show_type label

type label = Eps | Sym of char


We represent the non-deterministic transition relation $\delta$ by the following type:

In [3]:
#show_type trans_rel

type 'q trans_rel = ('q * label, 'q FSet.t) Dict.t


The type of the deterministic transition relation is `'q trans_rel_det`.  For the <span style="font-variant:small-caps;">Dfa</span>
produced in this notebook, the type parameter `'q` is instantiated with `'q FSet.t`, since the states of the <span style="font-variant:small-caps;">Dfa</span>
are sets of states of the <span style="font-variant:small-caps;">Nfa</span>.

In [4]:
#show_type trans_rel_det

type 'q trans_rel_det = ('q * char, 'q) Dict.t


A non-deterministic finite state machine has the following type:

In [5]:
#show_type nfa

type 'q nfa = {
  states : 'q FSet.t;
  sigma : char FSet.t;
  delta : 'q trans_rel;
  start : 'q;
  accepting : 'q FSet.t;
}


The deterministic finite state machines produced in this notebook have the type `'q FSet.t dfa`, where `dfa` is the following type.

In [6]:
#show_type dfa

type 'q dfa = {
  states : 'q FSet.t;
  sigma : char FSet.t;
  delta : 'q trans_rel_det;
  start : 'q;
  accepting : 'q FSet.t;
}


<hr style="height:5px;background-color:blue">

In order to present the construction of $\texttt{det}(F)$ we first have to define five auxiliary functions.
We start with the function `bigUnion`.  Given a set `m` that contains sets, the expression `bigUnion m`
returns the union of all sets in `m`, i.e. we have
$$ \texttt{bigUnion}(M) = \bigcup M = \bigl\{ x \bigm| \exists A \in M: x \in A \bigr\}. $$

Python computes this set with the *set comprehension* `{ x for A in M for x in A }`.  OCaml has no set
comprehensions, but the module `FSet` provides the operators `let*` and `return`:
```
let open FSet in
let* a = m in
let* x = a in
return x
```
is read as *"for every set `a` in `m` and every element `x` in `a`, collect `x`"*.  The expression `let open FSet in`
makes the operators of the module `FSet` available.

**Function `bigUnion m`**
- *Input:* `m` is a set of sets.
- *Output:* The union of all sets in `m`.

In [7]:
let bigUnion (m : 'a FSet.t FSet.t) : 'a FSet.t =
  let open FSet in
  let* a = m in
  let* x = a in
  return x

val bigUnion : 'a FSet.t FSet.t -> 'a FSet.t = <fun>


In [8]:
bigUnion (FSet.of_list [ FSet.of_list [1; 2; 3]; FSet.of_list [2; 3; 4]; FSet.of_list [3; 4; 5] ])

- : int FSet.t = {1, 2, 3, 4, 5}


The function `epsClosure` takes two arguments:
- `s` is a state, 
- `delta` is the transition function of the <span style="font-variant:small-caps;">Nfa</span> $F$.

The function computes the set of all those states that can be reached from the state
`s` via $\varepsilon$-transitions.
Formally, the set $\texttt{epsClosure}(q)$ is defined inductively:
- $s \in \texttt{epsClosure}(s)$.
- $p \in \texttt{epsClosure}(s) \wedge r \in \delta(p, \varepsilon) \;\rightarrow\; r \in \texttt{epsClosure}(s)$.
 
  If the state $p$ is an element of the $\varepsilon$-closure of the state $s$ 
  and there is an $\varepsilon$-transition from $p$ to some state $r$, then $r$ 
  is also an element of the $\varepsilon$-closure of $s$.
  
The implementation of `epsClosure` uses a *fixed-point algorithm*.  The `while` loop of Python is replaced
by the recursive function `loop`.  `Dict.get (q, Eps) ~default:FSet.empty delta` corresponds to
`δ.get((q, '𝜀'), set())`.

**Function `epsClosure s delta`**
- *Input:* `s` is a state of an <span style="font-variant:small-caps;">Nfa</span> and `delta` is its transition function.
- *Output:* The set of all states that can be reached from `s` via $\varepsilon$-transitions.

In [9]:
let epsClosure (s : 'q) (delta : 'q trans_rel) : 'q FSet.t =
  let rec loop (result : 'q FSet.t) : 'q FSet.t =
    let newStates = bigUnion (FSet.map (fun q -> Dict.get (q, Eps) ~default:FSet.empty delta) result) in
    if FSet.subset newStates result then result
    else loop (FSet.union result newStates)
  in
  loop (FSet.singleton s)

val epsClosure : 'q -> 'q trans_rel -> 'q FSet.t = <fun>


In order to transform an <span style="font-variant:small-caps;">Nfa</span> $F$ into a 
<span style="font-variant:small-caps;">Dfa</span>
$\texttt{det}(F)$ we have to extend the function $\delta:Q \times (\Sigma \cup \{\varepsilon\}) \rightarrow 2^Q$ into the function
$$\widehat{\delta}: Q \times \Sigma \rightarrow 2^Q. $$
The idea is that given a state $q$ and a character $c$,  the value of $\widehat{\delta}(q,c)$ is the set of all states that the
<span style="font-variant:small-caps;">Nfa</span> $F$ could reach when it reads the character $c$ in state $q$ and then performs an arbitrary number of $\varepsilon$-transitions.  Formally, the definition of $\widehat{\delta}$ is as follows:
$$ \widehat{\delta}(q_1, c) := \bigcup \bigl\{ \texttt{epsClosure}(q_2) \bigm| q_2 \in \delta(q_1, c) \bigr \}. $$
This formula is to be read as follows:
- For every state $q_2 \in Q$ that can be reached from the state $q_1$ by reading the character $c$ we
  compute $\texttt{epsClosure}(q_2)$.
- Then we take the union of all these sets $\texttt{epsClosure}(q_2)$.

The function $\widehat{\delta}$ is implemented as the function `deltaHat`, which takes three arguments:
- `s` is a state,
- `c` is a character,
- `delta` is the transition function of a 
  <span style="font-variant:small-caps;">Nfa</span>.

This function computes the set of all those states that can be reached 
from `s` when we first have a transition from state `s` to some state `p` 
on reading the character `c` followed by any number of $\varepsilon$-transitions
starting in `p`.  The formal definition of $\widehat{\delta}$ is the following: 

**Function `deltaHat s c delta`**
- *Input:* `s` is a state, `c` is a character, and `delta` is the transition function of an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* The set $\widehat{\delta}(s, c)$ of all states that can be reached from `s` by reading `c` followed by any number of $\varepsilon$-transitions.

In [10]:
let deltaHat (s : 'q) (c : char) (delta : 'q trans_rel) : 'q FSet.t =
  bigUnion (FSet.map (fun q -> epsClosure q delta) (Dict.get (s, Sym c) ~default:FSet.empty delta))

val deltaHat : 'q -> char -> 'q trans_rel -> 'q FSet.t = <fun>


The function  $\widehat{\delta}$ maps a state into a set of states.  Since the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ uses sets of states of the <span style="font-variant:small-caps;">Nfa</span> $F$ as its states we need a function that maps sets of states of the <span style="font-variant:small-caps;">Nfa</span> $F$ into sets of states.  Hence we generalize 
the function $\widehat{\delta}$ to the function
$$ \Delta: 2^Q \times \Sigma \rightarrow 2^Q $$
such that for a set $M$ of states and a character $c$ the expression $\Delta(M, c)$
computes the set of all those states that the <span style="font-variant:small-caps;">Nfa</span> $F$ could be in if it is in a state from the set $M$, then
reads the character $c$, and finally makes some $\varepsilon$-transitions.
The formal definition is as follows: 
$$ \Delta(M,c) := \bigcup \bigl\{ \widehat{\delta}(q,c) \bigm| q \in M \bigr\}. $$
This formula is easy to understand:  For every state  $q \in M$ we compute the set of states that the
<span style="font-variant:small-caps;">Nfa</span> $F$ could be in after reading the character $c$ and doing some 
$\varepsilon$-transitions.  Then we take the union of these sets.

**Function `capitalDelta m c delta`**
- *Input:* `m` is a set of states, `c` is a character, and `delta` is the transition function of an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* The set $\Delta(M, c)$, i.e. the union of the sets $\widehat{\delta}(q, c)$ for all $q \in M$.

In [11]:
let capitalDelta (m : 'q FSet.t) (c : char) (delta : 'q trans_rel) : 'q FSet.t =
  bigUnion (FSet.map (fun q -> deltaHat q c delta) m)

val capitalDelta : 'q FSet.t -> char -> 'q trans_rel -> 'q FSet.t = <fun>


The function `allStates` takes three arguments:
- $Q$ is $\texttt{epsClosure}(q_0)$, where $q_0$ is the start state of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$,
- $\delta$ is the transition function of the <span style="font-variant:small-caps;">Nfa</span> $F$, and
- $\Sigma$ is the alphabet of the <span style="font-variant:small-caps;">Nfa</span> $F$.

The function `allStates` computes the set of all states of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$
that can be reached from the start state.

**Function `allStates q delta sigma`**
- *Input:* `q` is the start state of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$, `delta` is the transition function of the <span style="font-variant:small-caps;">Nfa</span> $F$, and `sigma` is its alphabet.
- *Output:* The set of all states of $\texttt{det}(F)$ that can be reached from `q`.

In [12]:
let allStates (q : 'q FSet.t) (delta : 'q trans_rel) (sigma : char FSet.t) : 'q FSet.t FSet.t =
  let rec loop (result : 'q FSet.t FSet.t) : 'q FSet.t FSet.t =
    let newStates =
      let open FSet in
      let* m = result in
      let* c = sigma in
      return (capitalDelta m c delta)
    in
    if FSet.subset newStates result then result
    else loop (FSet.union result newStates)
  in
  loop (FSet.singleton q)

val allStates : 'q FSet.t -> 'q trans_rel -> char FSet.t -> 'q FSet.t FSet.t =
  <fun>


Now we are ready to formally define how the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$
is computed from the <span style="font-variant:small-caps;">Nfa</span>
$F = \bigl\langle Q, \Sigma, \delta, q_0, A \bigr\rangle$.
We define: 
$$ \texttt{det}(F) := \bigl\langle \texttt{allStates}(\texttt{epsClosure}(q_0)), \Sigma, \Delta, \texttt{epsClosure}(q_0), \widehat{A} \bigr\rangle $$
where the components of this tuple are given as follows:
- The set of states of $\texttt{det}(F)$ is the set of all states that can be reached from the set $\texttt{epsClosure}(q_0)$.
- The input alphabet $\Sigma$ does not change when going from $F$ to $\texttt{det}(F)$.
  After all, the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ has to recognize the same language as the non-deterministic
  <span style="font-variant:small-caps;">Nfa</span> $F$.
- The function $\Delta$, that has been defined previously, specifies how the set of states changes when a
  character is read.
- The start state $\texttt{epsClosure}(q_0)$ of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ is the set of all states
  that can be reached from the start state $q_0$ of the <span style="font-variant:small-caps;">Nfa</span> $F$
  via $\varepsilon$-transitions.
- The set of accepting states $\widehat{A}$ is the set of those subsets of $Q$ that contain an accepting
  state of the <span style="font-variant:small-caps;">Nfa</span> $F$:
  $$\widehat{A} := \bigl\{ M \in 2^Q \mid M \cap A \not= \{\} \bigr\}. $$

**Function `nfa2dfa nfa`**
- *Input:* `nfa` is an <span style="font-variant:small-caps;">Nfa</span> $F$.
- *Output:* The <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$, which accepts the same language as $F$.

In [13]:
let nfa2dfa (nfa : 'q nfa) : 'q FSet.t dfa =
  let ({ sigma; delta; start = q0; accepting = final; _ } : 'q nfa) = nfa in
  let newStart  = epsClosure q0 delta in
  let newStates = allStates newStart delta sigma in
  let newDelta  =
    Dict.of_list
      (List.concat_map (fun m -> List.map (fun c -> ((m, c), capitalDelta m c delta)) (FSet.to_list sigma))
                       (FSet.to_list newStates))
  in
  let newFinal = FSet.filter (fun m -> not (FSet.is_empty (FSet.inter m final))) newStates in
  { states = newStates; sigma; delta = newDelta; start = newStart; accepting = newFinal }

val nfa2dfa : 'q nfa -> 'q FSet.t dfa = <fun>


To test this function, use the notebook `02-Test-NFA-2-DFA.ipynb`.